# 01 – Data understanding: can we trust the data?

**Stage 3 of the project.** Before any analysis, we interrogate the data like a detective.
*Garbage in, garbage out*: if the data is wrong, the best model gives wrong answers.

**Five trust checks**
1. **Source**: who made the data, and how?
2. **Completeness**: are values missing?
3. **Uniqueness**: are rows duplicated?
4. **Validity**: do values obey physics?
5. **Consistency**: do related columns agree?

The written conclusions are in `docs/03_data_understanding.md`.

In [1]:
import sys
sys.path.append("..")          # lets Python find the src folder
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

df = pd.read_csv('../data/raw/ai4i2020.csv')
df.shape

(10000, 14)

Each row is **one production run** of the milling machine. 10,000 runs, 14 columns.

In [2]:
df.head()

,UDI,Product ID,Type,Air temperature [K],Process temperature [K],Rotational speed [rpm],Torque [Nm],Tool wear [min],Machine failure,TWF,HDF,PWF,OSF,RNF
0,1,M14860,M,298.1,308.6,1551,42.8,0,0,0,0,0,0,0
1,2,L47181,L,298.2,308.7,1408,46.3,3,0,0,0,0,0,0
2,3,L47182,L,298.1,308.5,1498,49.4,5,0,0,0,0,0,0
3,4,L47183,L,298.2,308.6,1433,39.5,7,0,0,0,0,0,0
4,5,L47184,L,298.2,308.7,1408,40.0,9,0,0,0,0,0,0


In [3]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 14 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   UDI                      10000 non-null  int64  
 1   Product ID               10000 non-null  str    
 2   Type                     10000 non-null  str    
 3   Air temperature [K]      10000 non-null  float64
 4   Process temperature [K]  10000 non-null  float64
 5   Rotational speed [rpm]   10000 non-null  int64  
 6   Torque [Nm]              10000 non-null  float64
 7   Tool wear [min]          10000 non-null  int64  
 8   Machine failure          10000 non-null  int64  
 9   TWF                      10000 non-null  int64  
 10  HDF                      10000 non-null  int64  
 11  PWF                      10000 non-null  int64  
 12  OSF                      10000 non-null  int64  
 13  RNF                      10000 non-null  int64  
dtypes: float64(3), int64(9), str(2)
me

## Checks 2 and 3: completeness and uniqueness

In [4]:
print("Missing values in the whole table:", df.isna().sum().sum())
print("Duplicate rows:", df.duplicated().sum())
print("UDI unique:", df["UDI"].is_unique, "| Product ID unique:", df["Product ID"].is_unique)

Missing values in the whole table: 0
Duplicate rows: 0
UDI unique: True | Product ID unique: True


## Check 4: validity – does the data obey physics?

In [5]:
df.describe().T

,count,mean,std,min,25%,50%,75%,max
UDI,10000.0,5000.50000,2886.895680,1.0,2500.75,5000.5,7500.25,10000.0
Air temperature [K],10000.0,300.00493,2.000259,295.3,298.30,300.1,301.50,304.5
Process temperature [K],10000.0,310.00556,1.483734,305.7,308.80,310.1,311.10,313.8
Rotational speed [rpm],10000.0,1538.77610,179.284096,1168.0,1423.00,1503.0,1612.00,2886.0
Torque [Nm],10000.0,39.98691,9.968934,3.8,33.20,40.1,46.80,76.6
Tool wear [min],10000.0,107.95100,63.654147,0.0,53.00,108.0,162.00,253.0
Machine failure,10000.0,0.03390,0.180981,0.0,0.00,0.0,0.00,1.0
TWF,10000.0,0.00460,0.067671,0.0,0.00,0.0,0.00,1.0
HDF,10000.0,0.01150,0.106625,0.0,0.00,0.0,0.00,1.0
PWF,10000.0,0.00950,0.097009,0.0,0.00,0.0,0.00,1.0


In [6]:
air = df["Air temperature [K]"]
process = df["Process temperature [K]"]
power = df["Torque [Nm]"] * df["Rotational speed [rpm]"] * 2 * np.pi / 60

print("Process always hotter than air:", (process > air).all())
print("Air temperature in °C:", round((air - 273.15).min(), 1), "to", round((air - 273.15).max(), 1))
print("Mechanical power in W:", round(power.min()), "to", round(power.max()))
print("Average process - air temperature:", round(process.mean() - air.mean(), 2), "K")

Process always hotter than air: True
Air temperature in °C: 22.2 to 31.4
Mechanical power in W: 1148 to 10470
Average process - air temperature: 10.0 K


The average difference is almost exactly **10 K**. Real factory data is rarely this neat:
a fingerprint of **synthetic** data. Results must be validated on real machine data before production use.

## How rare are failures?

In [7]:
counts = df["Machine failure"].value_counts()
print(counts)
print("Failure share: %.2f %%" % (100 * counts[1] / len(df)))
print("Accuracy of a model that ALWAYS says 'no failure': %.2f %%" % (100 * counts[0] / len(df)))

Machine failure
0    9661
1     339
Name: count, dtype: int64
Failure share: 3.39 %
Accuracy of a model that ALWAYS says 'no failure': 96.61 %


A useless model would be 96.6% accurate. **Accuracy is the wrong metric here.**

## Check 5: consistency – do the labels agree?

In [8]:
types = ["TWF", "HDF", "PWF", "OSF", "RNF"]
print(df[types].sum(), "\nTotal type marks:", df[types].sum().sum(), "| Failures:", counts[1])

TWF     46
HDF    115
PWF     95
OSF     98
RNF     19
dtype: int64 
Total type marks: 373 | Failures: 339


In [9]:
n_types = df[types].sum(axis=1)
print("Failure = 1 but no type marked:", ((df["Machine failure"] == 1) & (n_types == 0)).sum())
print("Type marked but failure = 0:   ", ((df["Machine failure"] == 0) & (n_types > 0)).sum())
print("More than one type at once:    ", (n_types > 1).sum())

inconsistent = df[(df["Machine failure"] == 0) & (n_types > 0)]
print("\nWhich types are the 'type but no failure' rows?")
print(inconsistent[types].sum())

Failure = 1 but no type marked: 9
Type marked but failure = 0:    18
More than one type at once:     24

Which types are the 'type but no failure' rows?
TWF     0
HDF     0
PWF     0
OSF     0
RNF    18
dtype: int64


**Puzzle solved:** 339 failures − 9 without a type + 18 random-failure marks without a failure
+ 25 extra marks from 24 multi-type runs = **373 type marks**.

**Decisions** (see `docs/03_data_understanding.md`): keep `Machine failure` as the official target,
make the failure-type model multi-label, do not model random failures, never use the type columns as inputs.

## Product variants

In [10]:
print((df["Type"].value_counts(normalize=True) * 100).round(1))
print()
print((df.groupby("Type")["Machine failure"].mean() * 100).round(2))

Type
L    60.0
M    30.0
H    10.0
Name: proportion, dtype: float64

Type
H    2.09
L    3.92
M    2.77
Name: Machine failure, dtype: float64


Low-quality products (L) fail almost twice as often as H. **Association, not yet a cause**: the EDA notebook finds out why.